In [3]:
# ====== 环境 ======
library(data.table)
library(mixtools)  # v1.2.0
library(ggplot2)
library(patchwork)

f_stat <- "/home/user/data3/lit/project/sORFs/10-feature-egi/processed/feature_preprare/peri/psite_frame_stats.v2.tsv"
f_rpf  <- "/home/user/data3/lit/project/sORFs/10-feature-egi/processed/feature_preprare/orf.rpf.psite.txt"

# ====== 读入与合并（只取必要列，节省内存）======
nThreads <- max(1, parallel::detectCores()-1)
dt1 <- fread(f_stat, sep="\t", header=TRUE, nThread=nThreads,
             select=c("ORF_id","frame0_fraction"))
dt2 <- fread(f_rpf,  sep="\t", header=TRUE, nThread=nThreads,
             select=c("ORF_id","Psites_codon_coverage"))
setkey(dt1, ORF_id); setkey(dt2, ORF_id)
dt <- dt1[dt2, nomatch=0L]; rm(dt1, dt2); gc()

# 仅 canonical ORF + 合法值
dt_canon <- dt[grepl("canonical", ORF_id, fixed=TRUE)]
dt_canon <- dt_canon[
  is.finite(frame0_fraction) & is.finite(Psites_codon_coverage) &
    frame0_fraction >= 0 & frame0_fraction <= 1 &
    Psites_codon_coverage >= 0 & Psites_codon_coverage <= 1
]

# 轻量预处理：winsorize，避免正好0/1导致方差塌陷
winsor <- function(x, eps=1e-6) pmax(eps, pmin(1-eps, x))
x_f0   <- winsor(dt_canon$frame0_fraction)
x_cov  <- winsor(dt_canon$Psites_codon_coverage)
    
# 抽样用于EM（几百万即可稳定，避免超大样本很慢）
samp <- function(x, n=2e6) if (length(x) > n) x[sample.int(length(x), n)] else x
x_f0_em  <- samp(x_f0)
x_cov_em <- samp(x_cov)

# ====== 阈值拟合：两分量高斯；等方差更稳健；失败则回退 ======
fit_threshold <- function(x){
  x <- x[is.finite(x)]
  init_mus <- quantile(x, c(0.25, 0.85), na.rm=TRUE)
  init_sds <- rep(sd(x), 2); init_lam <- c(0.5, 0.5)
  fm <- try(normalmixEM(x, k=2, mu=init_mus, sigma=init_sds, lambda=init_lam,
                        arbvar=FALSE,            # 等方差：避免sigma->0
                        maxrestarts=200,         # 增加重启次数
                        maxit=2000, epsilon=1e-8,
                        verb=FALSE), silent=TRUE)
  if (!inherits(fm, "try-error")) {
    hi <- which.max(fm$mu)
    thr <- fm$mu[hi] - 2*fm$sigma[hi]
    return(max(0, min(1, as.numeric(thr))))
  }
  # 回退：取右尾（>= 75%分位数）近似“高质量簇”
  gate <- quantile(x, 0.75, na.rm=TRUE)
  x_hi <- x[x >= gate]
  thr  <- mean(x_hi) - 2*sd(x_hi)
  max(0, min(1, as.numeric(thr)))
}

thr_f0  <- fit_threshold(x_f0_em)   # threshold for frame0_fraction
thr_cov <- fit_threshold(x_cov_em)  # threshold for Psites_codon_coverage

cat(sprintf("Thresholds (canonical only): frame0_fraction=%.4f, Psites_codon_coverage=%.4f\n",
            thr_f0, thr_cov))



,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,16081666,858.9,29289704,1564.3,17847414,953.2
Vcells,163841271,1250.1,501220703,3824.1,496567913,3788.6


number of iterations= 55 
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances is going to zero;  trying new starting values.
One of the variances i

Thresholds (canonical only): frame0_fraction=0.6783, Psites_codon_coverage=0.6642


ERROR while rich displaying an object: Error in Ops.data.frame(guide_loc, panel_loc): ‘==’ only defined for equally-sized data frames

Traceback:
1. tryCatch(withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler), error = outer_handler)
2. tryCatchList(expr, classes, parentenv, handlers)
3. tryCatchOne(expr, names, parentenv, handlers[[1L]])
4. doTryCatch(return(expr), name, parentenv, handler)
5. withCallingHandlers({
 .     if (!mime %in% names(repr::mime2repr)) 
 .         stop("No repr_* for mimetype ", mime, " in repr::mime2repr")
 .     rpr <- repr::mime2repr[[mime]](obj)
 .     if (is.null(rpr)) 
 .         return(NULL)
 .     prepare_content(is.raw(rpr), rpr)
 . }, error = error_handler)
6. repr::mime2repr[[mime]](obj)
7. repr_text.d

ERROR: Error in Ops.data.frame(guide_loc, panel_loc): ‘==’ only defined for equally-sized data frames


In [5]:
# ====== 作图：仅第一列（两张直方图 + 阈值虚线）======
make_hist <- function(x, thr, title){
  # 预先分箱（0~100，步长1），避免ggplot对巨量向量二次统计
  brks <- seq(0, 1, by=0.01)
  h <- hist(x, breaks=brks, plot=FALSE)
  df <- data.table(perc = head(brks,-1)*100, count = h$counts)
  ggplot(df, aes(perc, count)) +
    geom_col(width=1, fill="grey70") +
    geom_vline(xintercept = thr*100, linetype="dashed", color="red") +
    annotate("text", x = thr*100, y = max(df$count)*0.95,
             label = sprintf("%% threshold = %.1f", thr*100),
             hjust = -0.05, angle = 90, color="red", size=3) +
    labs(x="% ", y="Frequency", title=title) +
    theme_classic()
}

p_f0  <- make_hist(x_f0,  thr_f0,  "Known ORFs — % Reads in frame (frame0_fraction)")
p_cov <- make_hist(x_cov, thr_cov, "Known ORFs — % Uniformity (Psites_codon_coverage)")

# 分别输出两张图；也可上下拼接成一列
ggsave("canonical_frame0_fraction_hist.png", p_f0,  width=6, height=4, dpi=300)
ggsave("canonical_psites_codon_coverage_hist.png", p_cov, width=6, height=4, dpi=300)
# install.packages("cowplot")
library(cowplot)

combo <- plot_grid(p_f0 + theme(legend.position="none"),
                   p_cov + theme(legend.position="none"),
                   ncol = 1, align = "v", rel_heights = c(1,1))

ggsave("canonical_hist_column.png", combo, width = 6, height = 8, dpi = 300)



Attaching package: ‘cowplot’


The following object is masked from ‘package:patchwork’:

    align_plots




In [9]:
11945811/(11945811+533714+1737263)
1920522/(1920522+136130+689210)

[1] 0.8402609

[1] 0.6994241